[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C14_DL_Theory_Data_Course/04_synthetic_data/04_synthetic_data.ipynb)

# 04 · 合成数据与模型坍塌（用 numpy 复现）

目标：从零复现 **model collapse（递归自训练的方差几何收缩）**、**mode dropping**、**质量-多样性权衡**, 以及 **真实数据锚定 / 累积** 的缓解, 用 `assert` 钉死。

路线：一维方差收缩(对拍 ((N-1)/N)^g 理论) → N 越小坍塌越快 → 多模态 mode dropping → 替换 vs 累积对照 → 质量-多样性权衡 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊(自训练分布距离)。

> 心智模型：**反复用自己有限的输出重估自己, 估计偏差复利成分布退化**；解药 = **真实数据锚 + 累积(而非替换)**。

## 1 · model collapse 最小内核：方差几何收缩

最简递归自训练: 每代从上一代高斯采 N 个样本 → 重估高斯 → 作为下一代。

**有限 N 的样本方差系统性偏小**, 每代乘约 `(N-1)/N`, g 代后方差 `≈((N-1)/N)^g · σ₀² → 0`。多链平均测出这条几何衰减。

In [ ]:
import numpy as np

def collapse_chain(N=20, generations=10, var0=1.0, seed=0):
    '''递归自训练一条链, 返回各代方差。每代: 从当前高斯采 N 个 -> 重估均值/方差。'''
    rng = np.random.default_rng(seed)
    mu, var = 0.0, var0
    hist = [var]
    for _ in range(generations):
        sample = rng.normal(mu, np.sqrt(max(var, 0)), size=N)
        mu, var = sample.mean(), sample.var()      # 有偏估计(除以 N), 系统偏小
        hist.append(var)
    return np.array(hist)

N, G = 20, 10
chains = np.array([collapse_chain(N, G, seed=s) for s in range(500)])
mean_var = chains.mean(axis=0)
theory = np.array([((N - 1) / N) ** g for g in range(G + 1)])   # 理论几何衰减
print(f"{'代':>4}{'实测方差':>12}{'理论((N-1)/N)^g':>18}")
for g in range(G + 1):
    print(f'{g:>4}{mean_var[g]:>12.4f}{theory[g]:>18.4f}')

assert mean_var[0] == 1.0
assert np.all(np.diff(mean_var) <= 1e-9), '方差应逐代(几乎)单调收缩'
assert mean_var[-1] < 0.7, '若干代后方差明显收缩'
assert np.allclose(mean_var, theory, atol=0.03), '实测方差应吻合理论 ((N-1)/N)^g'
print('✅ MODEL COLLAPSE 内核: 方差按 ((N-1)/N)^g 几何收缩, 实测吻合理论')

## 2 · N 越小坍塌越快

每代样本数 `N` 是坍塌速度的关键: `N` 越小, 每代方差低估越多, 坍塌越快。
验证: 不同 `N` 下 g 代后的方差, `N` 越小越接近 0。`N→∞` 才不坍塌(但现实中 N 总有限)。

In [ ]:
G = 10
print(f"{'N(每代样本)':>12}{'10代后方差':>14}{'理论':>10}")
final_vars = {}
for N in [5, 20, 100, 1000]:
    chains = np.array([collapse_chain(N, G, seed=s) for s in range(500)])
    fv = chains.mean(axis=0)[-1]
    final_vars[N] = fv
    print(f'{N:>12}{fv:>14.4f}{((N-1)/N)**G:>10.4f}')

assert final_vars[5] < final_vars[20] < final_vars[100] < final_vars[1000], 'N 越小坍塌越严重'
assert final_vars[5] < 0.3, 'N=5 严重坍塌'
assert final_vars[1000] > 0.9, 'N=1000 几乎不坍塌(样本多)'
print('✅ N 越小坍塌越快; N→∞ 才不坍塌 —— 但现实中每代样本总是有限的')

## 3 · 早期坍塌：多模态的 mode dropping

坍塌**先丢尾部/罕见模式**。用 4 模式高斯混合: 递归自训练时, 小样本可能**没抽到**某个模式, 它就**永久消失**(不可逆棘轮)。
用 **mode coverage**(仍被覆盖的模式比例)度量多样性, 看它单调下降。

In [ ]:
centers = np.array([-6.0, -2.0, 2.0, 6.0])
MODE_STD = 0.5

def mode_coverage(samples, centers=centers, std=MODE_STD, thresh=2.0):
    '''有多少比例的模式至少有一个样本落在 thresh*std 内。'''
    return np.mean([np.any(np.abs(samples - c) < thresh * std) for c in centers])

def collapse_gmm(N=15, generations=8, seed=0):
    rng = np.random.default_rng(seed)
    comp = rng.integers(0, len(centers), 200)
    samples = rng.normal(centers[comp], MODE_STD)        # gen0: 真 4 模式
    cov = [mode_coverage(samples)]
    for _ in range(generations):
        idx = rng.integers(0, len(samples), N)           # 只看 N 个(小样本)
        samples = samples[idx] + rng.normal(0, MODE_STD, N)   # 重采(可能丢模式)
        cov.append(mode_coverage(samples))
    return np.array(cov)

covs = np.array([collapse_gmm(15, 8, seed=s) for s in range(400)]).mean(axis=0)
print('各代 mode coverage:', np.round(covs, 3))
assert covs[0] > 0.99, 'gen0 覆盖全部 4 个模式'
assert covs[-1] < covs[0], '坍塌后模式覆盖下降'
assert np.all(np.diff(covs) <= 1e-9), 'mode coverage 单调下降(模式丢了回不来)'
assert covs[-1] < 0.75, '若干代后丢掉至少一个模式'
print('✅ 早期坍塌 = mode dropping: 罕见模式先消失且不可逆, 多样性单调流失')

## 4 · 解药：累积(真实锚) vs 替换

**替换**(只用上一代合成): 方差几何收缩 -> 坍塌。
**累积**(真实数据 + 历代合成全留): 真实数据当锚, 每代把分布拉回 -> 不坍塌。
同一套代码, 唯一区别是「丢旧数据」还是「留旧数据 + 真实锚」。

In [ ]:
def accumulate_chain(N=20, generations=10, n_real=20, seed=0):
    '''累积: 真实数据锚 + 历代合成全部保留。'''
    rng = np.random.default_rng(seed)
    real = rng.normal(0, 1, n_real)              # 固定真实锚
    pool = list(real)
    mu, var = 0.0, 1.0
    hist = [float(np.var(pool))]
    for _ in range(generations):
        synth = rng.normal(mu, np.sqrt(max(var, 0)), N)   # 当前模型生成合成
        pool = pool + list(synth)                          # 累积(不丢真实/历史)
        arr = np.array(pool)
        mu, var = arr.mean(), arr.var()
        hist.append(var)
    return np.array(hist)

G = 10
rep = np.array([collapse_chain(20, G, seed=s) for s in range(400)]).mean(axis=0)
acc = np.array([accumulate_chain(20, G, seed=s) for s in range(400)]).mean(axis=0)
print(f"{'代':>4}{'替换(方差)':>14}{'累积(方差)':>14}")
for g in range(G + 1):
    print(f'{g:>4}{rep[g]:>14.4f}{acc[g]:>14.4f}')

assert rep[-1] < 0.7, '替换: 方差收缩(坍塌)'
assert acc[-1] > 0.85, '累积: 方差基本维持(不坍塌)'
assert acc[-1] > rep[-1], '累积明显优于替换'
print('✅ 解药: 累积(真实锚+保留历史)遏制坍塌; 替换则坍塌 —— 别丢真实数据、别只吃自己的输出')

## 5 · 质量-多样性权衡：过滤的代价

过滤能提质量(甚至反向救坍塌), 但**过滤本身制造分布偏移**: 收紧过滤 -> 平均质量↑、多样性↓。
这与生成评测的 **precision↑则 recall↓** 同源。玩具: 合成样本有(质量,位置)两维, 按质量阈值过滤, 看两条反向曲线。

In [ ]:
rng = np.random.default_rng(0)
M = 2000
# 合成样本: 位置 pos(多样性维), 质量 q(高质量样本多在中心, 尾部质量低)
pos = rng.uniform(-3, 3, M)
quality = np.exp(-0.3 * pos ** 2) + 0.1 * rng.standard_normal(M)   # 中心质量高、尾部低

def filter_and_measure(thresh):
    keep = quality >= thresh
    if keep.sum() == 0:
        return 0.0, 0.0, 0
    avg_q = quality[keep].mean()                 # 平均质量(保真)
    diversity = pos[keep].std()                  # 位置分散度(多样性)
    return avg_q, diversity, int(keep.sum())

print(f"{'质量阈值':>8}{'平均质量':>10}{'多样性':>10}{'保留数':>8}")
results = []
for th in [-0.5, 0.0, 0.3, 0.6, 0.9]:
    aq, dv, k = filter_and_measure(th)
    results.append((th, aq, dv))
    print(f'{th:>8.1f}{aq:>10.3f}{dv:>10.3f}{k:>8}')

qs = [r[1] for r in results]; dvs = [r[2] for r in results]
assert qs[-1] > qs[0], '过滤越严, 平均质量越高(保真↑)'
assert dvs[-1] < dvs[0], '过滤越严, 多样性越低(砍掉尾部)'
print('✅ 质量-多样性权衡: 收紧过滤 -> 质量↑但多样性↓ (precision↑则 recall↓ 的同源张力)')

---
## ✏️ 练习 1：合成生成 + 坍塌度量

实现 `collapse_speed(N, generations)`：跑方差收缩链(多链平均), 返回**方差降到初始一半所需的代数**(若 G 代内没降到一半返回 G)。
用它验证 `N` 越小, 半衰代数越短(坍塌越快)。

In [ ]:
def collapse_speed(N, generations=20, n_chains=300):
    # TODO: 多链平均得 mean_var(用 collapse_chain); 返回第一个使 mean_var[g] < 0.5 的 g
    #       (找不到则返回 generations)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
half_5 = collapse_speed(5)
half_50 = collapse_speed(50)
print(f'N=5  半衰代数 = {half_5}')
print(f'N=50 半衰代数 = {half_50}')
assert half_5 < half_50, 'N 越小坍塌越快(半衰代数越短)'
assert half_5 >= 1
# 理论: ((N-1)/N)^g = 0.5 -> g = ln0.5 / ln((N-1)/N)
import math
g_theory_5 = math.log(0.5) / math.log(4 / 5)
assert abs(half_5 - g_theory_5) <= 2, '应接近理论半衰代数'
print('✅ 练习 1 通过: 坍塌速度随 N 减小而加快, 吻合几何衰减理论')

## ✏️ 练习 2：坍塌度量(分布距离)

实现 `collapse_distance(N, generations)`：返回各代分布与**真分布 N(0,1)** 的距离(用均值差²+方差差², 即一维 Fréchet)。
验证距离随代数**单调增大**(分布越来越偏离真实)。

In [ ]:
def collapse_distance(N=20, generations=10, seed=0):
    # TODO: 跑一条 collapse_chain 但同时记录每代的 (mu, var);
    #       距离[g] = (mu_g - 0)^2 + (sqrt(var_g) - 1)^2   (一维 Fréchet 到 N(0,1))
    #       返回距离数组(长度 generations+1, 第0代距离≈0)
    #       提示: 自己重写一遍带 mu/var 记录的链
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
dists = np.array([collapse_distance(15, 10, seed=s) for s in range(300)]).mean(axis=0)
print('各代到真分布 N(0,1) 的距离:', np.round(dists, 4))
assert dists[0] < 0.05, '第0代就是真分布, 距离≈0'
assert dists[-1] > dists[0], '坍塌使分布越来越偏离真实'
assert dists[-1] > 0.05, '末代明显偏离'
print('✅ 练习 2 通过: 坍塌 = 与真分布距离单调增大')

## ✏️ 练习 3：多样性度量

实现 `diversity_entropy(samples, bins, range_)`：把样本分箱, 返回归一化的**香农熵**(覆盖越广越均匀, 熵越高 -> 多样性越高)。
验证: 单模式样本熵低, 均匀分布样本熵高。

In [ ]:
def diversity_entropy(samples, bins=20, range_=(-8, 8)):
    # TODO: np.histogram 分箱得计数 -> 概率 p; 香农熵 H=-sum(p log p)(只对 p>0);
    #       归一化: H / log(bins) (除以最大可能熵, 落在 [0,1])
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng = np.random.default_rng(0)
single_mode = rng.normal(0, 0.3, 2000)            # 集中在一处(低多样)
spread = rng.uniform(-8, 8, 2000)                 # 铺满(高多样)
h_single = diversity_entropy(single_mode)
h_spread = diversity_entropy(spread)
print(f'单模式熵 = {h_single:.3f}, 铺开熵 = {h_spread:.3f}')
assert 0 <= h_single <= 1 and 0 <= h_spread <= 1, '归一化熵在 [0,1]'
assert h_spread > h_single, '铺开(高多样)熵更高'
assert h_spread > 0.9, '近均匀分布熵接近1'
print('✅ 练习 3 通过: 熵量化多样性, 坍塌(集中)使熵下降')

## ✏️ 练习 4：过滤缓解(给坍塌注入外部信号)

STaR 思想: 过滤掉「坏」样本能阻止坍塌。实现 `filtered_chain(N, generations, keep_frac)`：
每代生成后, **只保留质量最高的 keep_frac 比例**(这里质量 = 离真均值0越近越好)再重估。
验证: 适度过滤比无过滤更能维持「接近真分布」(但注意过度过滤会伤多样性, 见 worked 5)。

In [ ]:
def filtered_chain(N=30, generations=10, keep_frac=0.7, seed=0):
    # TODO: 每代从当前高斯采 N 个; 按 |x - 0|(离真均值距离)排序, 保留最近的 keep_frac*N 个;
    #       用保留的重估 mu,var; 返回各代 |mu_g - 0| (到真均值的偏移)
    #       keep_frac=1.0 等价无过滤
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
drift_nofilter = np.array([filtered_chain(30, 10, 1.0, s) for s in range(300)]).mean(0)
drift_filter   = np.array([filtered_chain(30, 10, 0.7, s) for s in range(300)]).mean(0)
print(f'无过滤   末代均值偏移 = {drift_nofilter[-1]:.4f}')
print(f'过滤0.7  末代均值偏移 = {drift_filter[-1]:.4f}')
# 向真均值过滤把分布往中心拉, 抑制随机漂移
assert drift_filter[-1] <= drift_nofilter[-1] + 0.02, '向真值过滤应抑制均值漂移'
assert drift_filter[0] < 0.1, '第0代偏移小'
print('✅ 练习 4 通过: 过滤(外部信号)能抑制漂移; 但记住 worked5 —— 过度过滤伤多样性')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def collapse_speed(N, generations=20, n_chains=300):
    chains = np.array([collapse_chain(N, generations, seed=s) for s in range(n_chains)])
    mv = chains.mean(axis=0)
    for g in range(len(mv)):
        if mv[g] < 0.5:
            return g
    return generations

In [ ]:
# 练习 2 参考答案
def collapse_distance(N=20, generations=10, seed=0):
    rng = np.random.default_rng(seed)
    mu, var = 0.0, 1.0
    dists = [(mu - 0) ** 2 + (np.sqrt(var) - 1) ** 2]
    for _ in range(generations):
        s = rng.normal(mu, np.sqrt(max(var, 0)), N)
        mu, var = s.mean(), s.var()
        dists.append((mu - 0) ** 2 + (np.sqrt(max(var, 0)) - 1) ** 2)
    return np.array(dists)

In [ ]:
# 练习 3 参考答案
def diversity_entropy(samples, bins=20, range_=(-8, 8)):
    counts, _ = np.histogram(samples, bins=bins, range=range_)
    p = counts / counts.sum()
    p = p[p > 0]
    H = -np.sum(p * np.log(p))
    return H / np.log(bins)

In [ ]:
# 练习 4 参考答案
def filtered_chain(N=30, generations=10, keep_frac=0.7, seed=0):
    rng = np.random.default_rng(seed)
    mu, var = 0.0, 1.0
    drift = [abs(mu - 0)]
    k = max(int(keep_frac * N), 2)
    for _ in range(generations):
        s = rng.normal(mu, np.sqrt(max(var, 0)), N)
        order = np.argsort(np.abs(s - 0.0))      # 离真均值近的优先
        kept = s[order[:k]]
        mu, var = kept.mean(), kept.var()
        drift.append(abs(mu - 0))
    return np.array(drift)

---
## 🧪 真实数据胶囊：真实分布上的自训练坍塌

用真实数据的统计量(尝试联网拉 **加州房价某特征**, 失败回退到内置真实统计量)做递归自训练, 量化合成分布如何逐代偏离真实。真实分布上 model collapse 同样发生。

In [ ]:
def load_real_feature(n=500, seed=0):
    '''尝试 sklearn 加州房价的 MedInc(收入)特征; 失败回退到其真实统计量构造。'''
    try:
        from sklearn.datasets import fetch_california_housing
        x = fetch_california_housing().data[:, 0]      # MedInc
        rng = np.random.default_rng(seed)
        x = rng.choice(x, size=n, replace=False)
        return x, 'sklearn 加州房价 MedInc(真实)'
    except Exception:
        # 回退: MedInc 真实统计量 (均值~3.87, 标准差~1.9, 右偏)
        rng = np.random.default_rng(seed)
        x = rng.lognormal(mean=1.2, sigma=0.5, size=n)  # 近似右偏收入分布
        return x, '内置真实统计量回退(右偏收入)'

real, src = load_real_feature(n=500)
real = (real - real.mean()) / real.std()              # 标准化
print(f'数据来源: {src}, n={len(real)}, 真实方差={real.var():.3f}')

def selftrain_on_real(real, N=30, generations=8, seed=0):
    '''从真实数据出发递归自训练(替换设定), 记录每代方差。'''
    rng = np.random.default_rng(seed)
    samples = rng.choice(real, size=N, replace=True)   # gen1 从真实采
    var_hist = [real.var(), samples.var()]
    mu, var = samples.mean(), samples.var()
    for _ in range(generations - 1):
        samples = rng.normal(mu, np.sqrt(max(var, 0)), N)   # 之后只用合成(替换)
        mu, var = samples.mean(), samples.var()
        var_hist.append(var)
    return np.array(var_hist)

vh = np.array([selftrain_on_real(real, 30, 8, seed=s) for s in range(300)]).mean(axis=0)
print('各代方差(真实->递归合成):', np.round(vh, 3))
assert vh[-1] < vh[0], '递归自训练使方差偏离真实(收缩)'
assert vh[-1] < 0.85 * vh[0], '若干代后明显坍塌'
print('✅ 真实分布上 model collapse 同样发生: 递归(替换)自训练使方差收缩')

**🧪 胶囊练习**：实现 `collapse_ratio(var_hist)`：返回末代方差 / 初代(真实)方差(越小坍塌越重)。

In [ ]:
def collapse_ratio(var_hist):
    # TODO: 返回 var_hist[-1] / var_hist[0]
    raise NotImplementedError

In [ ]:
# 自测
ratio = collapse_ratio(vh)
assert 0 < ratio < 1, '坍塌使比值 <1'
print(f'坍塌比 = 末代/真实方差 = {ratio:.2f} (越小坍塌越重)')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def collapse_ratio(var_hist):
    return var_hist[-1] / var_hist[0]

### 小结
- **合成数据**是双刃剑: 缓解数据稀缺/做蒸馏, 但递归使用会触发 **model collapse**(递归的诅咒)。
- 内核: 用自己有限的输出反复重估自己, **方差按 ((N-1)/N)^g 几何收缩**; N 越小坍塌越快。
- **早期坍塌 = mode dropping**(罕见模式先丢且不可逆), 单看保真度察觉不到, 必须看多样性。
- 解药: **真实数据锚定 + 累积(而非替换)** 遏制坍塌; **过滤**(注入外部信号)也能救, 但**过度过滤伤多样性**(质量-多样性权衡)。
- 评测合成数据(及任何生成模型)**永远至少分保真+多样两维**; 盯住多样性指标才能早期发现坍塌。

下一站: **模块 05 · 生成媒体评测** —— 把「保真 vs 多样」做成可计算的 FID、IS、precision-recall。